# 워크북 3 — 시설관리 AI 신고 분류 및 대응 안내 (student)
학습 질문: 같은 인터넷 장애를 서로 다르게 말해도 AI가 담당 분야를 찾아낼까?
CPU 실행, 가상 데이터, 외부 API·실제 신고 전송 없음.

## 1. 데이터 확인
45건: 학습 36, 독립 평가 9. 입력 text, 정답 label, split. Drive 파일 미업로드 시 내장 복사본 사용.

In [ ]:
import pandas as pd
import json
from io import StringIO
from pathlib import Path
# Drive에서 받은 파일을 Colab 파일 창에 업로드하면 우선 사용합니다.
CSV_BACKUP = 'id,label,text,split\nT01,전산,강의실 인터넷 연결이 안 됩니다,train\nT02,전산,컴퓨터 화면이 멈췄어요,train\nT03,전산,와이파이 접속이 끊깁니다,train\nT04,전산,PC 전원이 켜지지 않습니다,train\nT05,전산,프린터 출력이 되지 않아요,train\nT06,전산,네트워크 속도가 느립니다,train\nT07,전산,컴퓨터 로그인 화면에서 멈춥니다,train\nT08,전산,인터넷 접속 오류가 발생합니다,train\nT09,전산,노트북 무선 연결이 불안정해요,train\nT10,전산,모니터 화면이 나오지 않습니다,train\nT11,전산,프린터 용지 오류가 표시됩니다,train\nT12,전산,PC 프로그램이 작동하지 않아요,train\nT13,환경,강의실 쓰레기통이 가득 찼어요,train\nT14,환경,복도 바닥에 먼지가 많습니다,train\nT15,환경,교실 책상에 쓰레기가 있습니다,train\nT16,환경,화장실 청소가 필요합니다,train\nT17,환경,분리수거함이 넘칩니다,train\nT18,환경,바닥에 종이컵이 흩어졌어요,train\nT19,환경,복도 청소를 요청합니다,train\nT20,환경,창틀에 먼지가 쌓였습니다,train\nT21,환경,쓰레기통을 비워 주세요,train\nT22,환경,강의실 바닥에 오염이 있습니다,train\nT23,환경,책상 주변이 더러워요,train\nT24,환경,분리수거함 주변에 쓰레기가 많아요,train\nT25,설비,강의실 에어컨이 작동하지 않습니다,train\nT26,설비,천장 조명이 켜지지 않아요,train\nT27,설비,수도꼭지에서 물이 샙니다,train\nT28,설비,난방이 되지 않습니다,train\nT29,설비,교실 형광등이 깜빡입니다,train\nT30,설비,화장실 물이 내려가지 않아요,train\nT31,설비,에어컨 바람이 나오지 않습니다,train\nT32,설비,강의실 조명이 어둡습니다,train\nT33,설비,냉방 장치가 고장 났어요,train\nT34,설비,세면대 배수가 막혔습니다,train\nT35,설비,난방 온도가 너무 낮아요,train\nT36,설비,수도에서 물이 계속 떨어집니다,train\nE01,전산,인터넷 와이파이 접속이 안 돼요,test\nE02,전산,컴퓨터 모니터 화면이 검게 나옵니다,test\nE03,전산,프린터 출력 오류가 있습니다,test\nE04,환경,복도에 쓰레기가 가득합니다,test\nE05,환경,강의실 바닥 먼지 청소 부탁해요,test\nE06,환경,분리수거함을 비워 주세요,test\nE07,설비,에어컨 냉방이 작동하지 않아요,test\nE08,설비,천장 조명이 깜빡여요,test\nE09,설비,세면대 수도 물이 샙니다,test\n'
KB_BACKUP = [{'label': '전산', 'source': '가상 매뉴얼 IT-01', 'text': '전산 담당자에게 장소와 장비 종류, 화면의 오류 메시지를 전달합니다.'}, {'label': '환경', 'source': '가상 매뉴얼 CL-01', 'text': '환경 담당자에게 장소와 청소 필요 상태를 알립니다.'}, {'label': '설비', 'source': '가상 매뉴얼 FM-01', 'text': '시설 담당자에게 장소와 증상을 알리고 점검을 요청합니다. 직접 수리하지 않습니다.'}]
path = Path('facility_reports.csv')
df = pd.read_csv(path if path.exists() else StringIO(CSV_BACKUP))
kp = Path('facility_knowledge.json')
kb = json.loads(kp.read_text(encoding='utf-8')) if kp.exists() else KB_BACKUP
assert len(df)==45 and set(df['split'])=={'train','test'}
train = df[df.split=='train'].copy()
test = df[df.split=='test'].copy()
assert set(train.text).isdisjoint(test.text)
print('학습',len(train),'평가',len(test))
display(df.groupby(['split','label']).size())

## 2. AI 이전: 키워드 규칙

In [ ]:
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
keywords = {'전산':['인터넷','컴퓨터','프린터'], '환경':['쓰레기','청소'], '설비':['에어컨','수도','조명']}
def rule(text):
    scores = {label:sum(k in text for k in words) for label,words in keywords.items()}
    best=max(scores,key=scores.get)
    return best if scores[best] else '미확정'
base_pred=test.text.map(rule)
print('규칙 정확도:', accuracy_score(test.label,base_pred))

## 3. 지도학습: TF-IDF + 로지스틱 회귀
규칙과 같은 평가 9건을 비교합니다.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
model = make_pipeline(TfidfVectorizer(analyzer='char',ngram_range=(2,4)), LogisticRegression(max_iter=500,random_state=42))
model.fit(train.text,train.label)
pred=model.predict(test.text)
print('AI 정확도:',accuracy_score(test.label,pred))
results=test[['text','label']].copy()
results['rule']=base_pred
results['AI']=pred
display(results)
ConfusionMatrixDisplay.from_predictions(test.label,pred,display_labels=['전산','환경','설비'],labels=['전산','환경','설비'])
plt.show()

## 4. 대응 안내: 점수가 낮으면 미확정, 높으면 가상 매뉴얼에서 근거 반환
점수는 정확성을 보장하지 않으며 실제 담당자에게 전송하지 않습니다.

In [ ]:
MIN_SCORE=0.44
def route(text):
    text=text.strip()
    if not text:
        return {'신고':text,'분류':'미확정','점수':0.0,'안내':'담당자 확인 필요','출처':'없음'}
    vec=model[0].transform([text])
    probs=model.predict_proba([text])[0]
    score=float(probs.max())
    label=model.classes_[probs.argmax()]
    if vec.nnz==0 or score<MIN_SCORE:
        return {'신고':text,'분류':'미확정','점수':round(score,3),'안내':'담당자 확인 필요','출처':'없음'}
    hit=next(r for r in kb if r['label']==label)
    return {'신고':text,'분류':label,'점수':round(score,3),'안내':hit['text'],'출처':hit['source']}
for q in ['인터넷 연결이 안 돼요','쓰레기통이 가득 찼어요','에어컨이 작동하지 않아요','오늘 점심 메뉴는?']:
    print(route(q))

## 5. 신고 예제 입력
제공된 예제 문장 4개를 그대로 실행하고 분류 및 점수, 안내, 출처를 확인하세요. 개인정보·긴급 사건은 입력하지 않습니다.

In [ ]:
my_reports = ['강의실 인터넷이 끊겨요','복도 쓰레기를 치워 주세요','조명이 켜지지 않아요','오늘 점심은?']
my_results = pd.DataFrame([route(q) for q in my_reports])
display(my_results)
my_results.to_csv('facility_results.csv',index=False,encoding='utf-8-sig')

## 6. PC·스마트폰 시설관리 신고 앱 (선택)
6-0 설치 → 6-1 실행 → 공유 QR/주소로 스마트폰 접속. PC Colab은 연결 유지.
기본 녹음 앱에서 아래 가상의 신고를 한 문장씩 녹음하고 파일에 저장합니다.
- 인터넷 연결이 안 돼요.
- 쓰레기통이 가득 찼어요.
- 에어컨이 작동하지 않아요.
앱의 파일 선택 버튼에서 M4A/WAV/MP3 선택 → 음성→텍스트 → 교정 → 신고 분야 분류 → 결과 CSV 다운로드.
사진은 선택 증거로 업로드하고 관측 내용을 직접 작성합니다. 사진 자동 인식은 하지 않습니다.
텍스트 질문은 모델 다운로드 없이 가능하며 음성 STT 최초 실행은 Whisper 다운로드가 필요합니다. 공개 공유 링크에 개인정보를 올리지 마세요.

In [ ]:
# 6-0. 모바일 앱 설치 (처음 1회만)
%pip install -q "gradio>=5,<7" "qrcode[pil]" transformers


In [ ]:
# 6-1. PC / 스마트폰 시설관리 신고 앱
import gradio as gr
import qrcode
import subprocess
from pathlib import Path
from uuid import uuid4
from tempfile import TemporaryDirectory, mkdtemp
from IPython.display import display, HTML
from html import escape

_stt = None
AUDIO_EXT = {'.wav','.m4a','.mp3','.aac','.webm','.ogg','.amr','.flac','.3gp','.mp4'}

def transcribe_file(path):
    global _stt
    if not path:
        return '', '파일을 먼저 선택하세요.'
    if Path(path).suffix.lower() not in AUDIO_EXT:
        return '', 'M4A, WAV, MP3 등 녹음 파일을 선택하세요.'
    try:
        with TemporaryDirectory() as folder:
            wav = Path(folder) / 'report.wav'
            subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(path),'-vn','-ac','1','-ar','16000','-c:a','pcm_s16le',str(wav)],check=True,capture_output=True,timeout=60)
            if _stt is None:
                from transformers import pipeline
                import torch
                _stt = pipeline('automatic-speech-recognition',model='openai/whisper-small',device=0 if torch.cuda.is_available() else -1)
            result = _stt(str(wav),generate_kwargs={'language':'korean','task':'transcribe'})['text'].strip()
        return result, '전사 완료. 문장을 확인·교정한 뒤 신고 분류를 누르세요.' if result else '인식된 문장이 없습니다. 다시 녹음하거나 직접 입력하세요.'
    except Exception as e:
        return '', f'음성 처리 실패: {type(e).__name__}. 파일을 다시 선택하거나 문장을 직접 입력하세요.'

def classify_report(text, observation, photo, audio_file):
    text = (text or '').strip()
    observation = (observation or '').strip()
    if not text:
        return '미확정', 0.0, '신고 문장을 입력하거나 음성 전사 후 교정하세요.', '없음', None
    # 사람이 관측한 사진 설명을 선택적으로 결합. 사진 자동 진단은 하지 않음.
    combined = text + (' ' + observation if observation else '')
    result = route(combined)
    row = dict(result, 음성파일=Path(audio_file).name if audio_file else '', 사진파일=Path(photo).name if photo else '', 사진관측=observation)
    out = Path(mkdtemp(prefix='facility_result_')) / 'facility_mobile_result.csv'
    pd.DataFrame([row]).to_csv(out,index=False,encoding='utf-8-sig')
    return result['분류'], result['점수'], result['안내'], result['출처'], str(out)

with gr.Blocks(title='13주 시설관리 AI 신고') as demo:
    gr.Markdown('# 🏫 시설관리 AI 신고 실습\n실제 신고 전송 없음. 개인정보를 올리지 마세요.')
    gr.Markdown('**녹음 예시:** “인터넷 연결이 안 돼요.” / “쓰레기통이 가득 찼어요.” / “에어컨이 작동하지 않아요.”')
    voice = gr.File(label='① 📁 파일 선택 (녹음 M4A/WAV/MP3)',file_count='single',type='filepath',file_types=None)
    gr.Markdown('기본 녹음 앱에서 저장하고 파일/찾아보기/내 파일에서 선택합니다. 직접 입력도 가능합니다.')
    transcribe = gr.Button('② 음성→텍스트',variant='primary')
    status = gr.Textbox(label='음성 처리 상태',interactive=False)
    report = gr.Textbox(label='③ 신고 문장 확인·교정 또는 직접 입력',lines=3)
    with gr.Accordion('선택: 시설물 사진 증거 추가',open=False):
        photo = gr.Image(label='시설물 사진 업로드',sources=['upload'],type='filepath')
        observation = gr.Textbox(label='사람이 사진에서 확인한 관측 사실',lines=2)
        gr.Markdown('사진은 증거이며 자동 영상분류나 고장 진단을 하지 않습니다.')
    go = gr.Button('④ 신고 분야 분류 및 안내',variant='primary')
    label = gr.Textbox(label='분류',interactive=False)
    score = gr.Number(label='분류 점수',interactive=False)
    guide = gr.Textbox(label='대응 안내',lines=3,interactive=False)
    source = gr.Textbox(label='근거 출처',interactive=False)
    download = gr.File(label='⑤ 결과 CSV 다운로드',interactive=False)
    transcribe.click(transcribe_file,voice,[report,status],concurrency_limit=1)
    go.click(classify_report,[report,observation,photo,voice],[label,score,guide,source,download])

demo.launch(share=True,prevent_thread_lock=True)
url = getattr(demo,'share_url',None)
if url:
    print('스마트폰 접속:',url)
    display(HTML(f'<a href="{escape(url,quote=True)}" target="_blank">PC/스마트폰 접속 링크</a>'))
    try: display(qrcode.make(url).get_image())
    except Exception: print('QR 표시 실패. 위 주소 이용.')
else:
    print('공유 주소 생성 실패. Colab 런타임 유지 및 네트워크 확인. 단계 5의 텍스트 실습은 가능.')


## 7. 학생 기록 — 출력 결과만 기록
2번 규칙 정확도, 3번 AI 정확도, 5번 결과표의 신고 문장·분류·점수·안내·출처를 그대로 적습니다. facility_results.csv를 내려받습니다.
6-1 앱을 사용했다면 화면에 표시된 전사 문장·분류·점수·안내·출처를 기록하고 facility_mobile_result.csv와 화면을 첨부합니다. 선택 실습을 하지 않으면 미실행으로 표시합니다.